In [3]:
"""Qubit mapper usage examples."""

# --------------------------------------------------------------------------------------------
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License. See LICENSE.txt in the project root for license information.
# --------------------------------------------------------------------------------------------

################################################################################
# start-cell-create
import numpy as np
from qdk.openqasm import circuit
from qdk.widgets import Circuit as CircuitWidget

from qiskit import QuantumCircuit, qasm3
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import QubitHamiltonian
from qdk_chemistry.data.circuit import Circuit as DataCircuit

# Build a simple qubit Hamiltonian as a sum of 5 Pauli strings
qubit_hamiltonian = QubitHamiltonian(
    pauli_strings=["ZZI", "XXI"],
    coefficients=np.array([0.5, 0.5]),
)
print(f"Qubit Hamiltonian has {qubit_hamiltonian.num_qubits} qubits")
print("Pauli strings:", qubit_hamiltonian.pauli_strings)
# end-cell-create
################################################################################

# Create a trivial (identity) state preparation circuit matching the Hamiltonian size
empty_qc = QuantumCircuit(qubit_hamiltonian.num_qubits)
state_preparation = DataCircuit(qasm=qasm3.dumps(empty_qc))

evolution_builder = create("hamiltonian_unitary_builder", "trotter", order=2)
circuit_mapper = create("controlled_evolution_circuit_mapper", "pauli_sequence")

from qdk_chemistry.data.controlled_unitary import ControlledUnitary

# Build the Trotter time-evolution unitary
time_evol_unitary = evolution_builder.run(qubit_hamiltonian, time=1)

# Wrap as a controlled unitary
ctrl_time_evol = ControlledUnitary(
    time_evolution_unitary=time_evol_unitary,
    control_indices=[0],
)

# Map to a circuit with power=1
ctrl_time_evol_circuit = circuit_mapper.run(ctrl_time_evol)

# Display the controlled time evolution circuit
display(CircuitWidget(circuit(ctrl_time_evol_circuit.get_qasm())))

Qubit Hamiltonian has 3 qubits
Pauli strings: ['ZZI', 'XXI']
